In [ ]:
'''
이건 그냥 안 하고, 바로 입력 테스트케이스 생성하는 게 나을 것 같아서 7번으로 넘어감

In [ ]:
'''
이제 검증된 problem, solution, testcase들을 확보했는데, 구체적으로 테스트 케이스들의 타입을 분류해보려고 해. 크게 4가지 타입인데, 내용은 다음과 같아.
-----------------
### 1) 정답성 검증
기본 예제, 소형 입력, 사람이 손으로 검산 가능한 케이스
### 2) 엣지 케이스 검증
경계값, 최소/최대, 빈 구조, 중복, 정렬 여부, 음수/0, 단일 원소, 동일값 반복 등
### 3) 반례 탐지
많이 틀리는 구현을 깨는 케이스
예:
- 오버플로우
- 인덱스 실수
- 정렬 기준 실수
- `<=` / `<` 조건 차이
- 중복 처리 실수
- 방문 처리 시점 오류
- greedy 오판
- DP 초기값 오류
- 이분탐색 경계 실수
- union-find parent 갱신 실수
- 그래프에서 disconnected 미처리
### 4) 성능 검증
제한시간에 근접한 대형 입력
예:
- 최댓값 크기
- 편향된 입력
- worst-case 패턴
- 시간복잡도 `O(N^2)` 풀이가 터지는 구조
- 재귀 깊이 위험
- hash collision 유사 상황
- 정렬/우선순위큐 연산이 극대화되는 패턴
-----------------
gemma 4 31B api를 호출해서(2026년 4월 출시된 모델) 문제 내용과 테스트케이스들을 모두 주고 각각을 아까 테스트 케이스 4개 중에 하나로 분류하게 만드는 거야. validated_problems.csv, validated_problem_testcases.csv에서 문제와 테스트케이스들을 읽어온 후, testcases 테이블에 type 열을 하나 추가해서 거기에 4개 중에 하나를 적은 다음 저장하게 하면 되겠지. 물론 testcase 파일은 크기가 크니까 배치 10000 단위로 읽게 하고. 이해 됐어? 모르겠는거 질문해줘

In [ ]:
import os
api_key = os.environ.get("GEMINI_API_KEY")

In [3]:
import os
import time
import json
import random
import pandas as pd
from google import genai
from google.genai import types

# ---------------------------------------------------------
# [설정값]
# ---------------------------------------------------------
# api_key 변수는 이미 선언되어 있다고 가정합니다. (예: api_key = "AIzaSy...")
client = genai.Client(api_key=api_key)
MODEL_NAME = 'gemma-4-31b-it'

PROBLEMS_FILE = 'validated_problems.csv'
TESTCASES_FILE = 'validated_problem_testcases.csv'
OUTPUT_FILE = 'typed_problem_testcases.csv'

BATCH_SIZE = 50  # 한 번의 API 호출에 묶어 보낼 테스트케이스 수

# [분산 처리용 범위 설정] (1-based index)
# 예: 컴퓨터 A는 1~1500, 컴퓨터 B는 1501~3000 식으로 나누어 작업할 수 있습니다.
START_IDX = 1       # 시작 문제 순서 (1부터 시작)
END_IDX = 4529      # 끝 문제 순서

# ---------------------------------------------------------
# [유틸리티 함수]
# ---------------------------------------------------------
def classify_testcases_with_retry(problem_context, testcases_batch, max_retries=3):
    """
    API를 호출하고 에러 시 Exponential Backoff 방식으로 재시도합니다.
    """
    prompt = f"""
[Problem Description]
{problem_context}

[Test Case Classification Criteria]
1) base: Basic examples, small inputs, cases that can be manually verified.
2) edge: Boundary values, min/max, empty structures, duplicates, sorted/unsorted, negative/zero, single element, repeated identical values, etc.
3) counter: Cases designed to break common incorrect implementations. Examples: overflow, index errors, wrong sorting criteria, <= vs < condition errors, duplicate handling errors, visited check timing errors, greedy misjudgments, DP initial value errors, binary search boundary errors, union-find parent update errors, unhandled disconnected graphs.
4) performance: Large inputs near the time limit. Examples: max size values, skewed inputs, worst-case patterns, structures that break O(N^2) time complexity, recursion depth risks, hash collision-like situations, patterns maximizing sort/priority queue operations.

[Test Cases to Evaluate (Total: {len(testcases_batch)})]
{json.dumps(testcases_batch, ensure_ascii=False, indent=2)}

Analyze the input/output formats and lengths of the above test cases, and classify each into exactly one of the four types: "base", "edge", "counter", or "performance".
Your response MUST be a pure JSON object in the exact format shown below. Do NOT add any other explanations or markdown formatting outside the JSON.
{{
    "testcase_order_1": "base",
    "testcase_order_2": "performance",
    ...
}}
"""
    
    for attempt in range(max_retries):
        try:
            # JSON 응답 강제 (Structured Output)
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    temperature=0.1 # 분류의 일관성을 위해 낮은 온도 설정
                )
            )
            
            text = response.text.strip()
            
            # 마크다운 포맷(```json ... ```)이 섞여올 경우를 대비한 방어 로직
            if text.startswith('```json'):
                text = text.replace('```json', '').replace('```', '').strip()
            elif text.startswith('```'):
                text = text.replace('```', '').strip()
                
            return json.loads(text)
            
        except Exception as e:
            if attempt == max_retries - 1:
                print(f"      ❌ [실패] 최대 재시도 횟수 초과. 에러: {e}")
                # 판단 불가(최종 실패) 시 해당 배치의 모든 테스트케이스 타입을 "ERROR"로 지정
                return {str(tc['testcase_order']): "ERROR" for tc in testcases_batch}
                
            sleep_time = (2 ** attempt) + random.uniform(0, 1)
            print(f"      ⚠️ API 호출 에러({e}). {sleep_time:.1f}초 후 재시도합니다...")
            time.sleep(sleep_time)


# ---------------------------------------------------------
# [메인 파이프라인]
# ---------------------------------------------------------
print("🚀 [Step 1] 문제 및 테스트케이스 데이터 로드 중...")
df_problems = pd.read_csv(PROBLEMS_FILE)

# 파일이 이미 존재하면 진행 상황을 이어서 하기 위해 OUTPUT_FILE을 로드
if os.path.exists(OUTPUT_FILE):
    print(f"📂 기존 작업 파일({OUTPUT_FILE})을 이어서 불러옵니다.")
    df_testcases = pd.read_csv(OUTPUT_FILE)
else:
    print(f"📄 원본 테스트케이스 파일({TESTCASES_FILE})을 새로 불러옵니다.")
    df_testcases = pd.read_csv(TESTCASES_FILE)
    # type 열이 없다면 None으로 초기화
    if 'type' not in df_testcases.columns:
        df_testcases['type'] = None

# [추가된 방어 로직] 데이터 타입 불일치(예: 정수 2 vs 실수 2.0 vs 문자열 '2')로 인한 매칭 실패를 원천 차단
df_problems['id'] = df_problems['id'].astype(str).str.replace(r'\.0$', '', regex=True)
df_testcases['problem_id'] = df_testcases['problem_id'].astype(str).str.replace(r'\.0$', '', regex=True)

# 작업 범위(Index) 필터링 로직
start_idx = max(0, START_IDX - 1)
end_idx = min(len(df_problems), END_IDX)
target_problems = df_problems.iloc[start_idx:end_idx]

problem_dict = dict(zip(target_problems['id'], target_problems['question']))
print(f"✅ 총 {len(df_problems)}문제 중 {START_IDX}번째 ~ {end_idx}번째 문제(총 {len(target_problems)}개) 작업을 시작합니다.\n")

print("🚀 [Step 2] 테스트케이스 분류 작업 시작...")

for idx, (prob_id, question_text) in enumerate(problem_dict.items(), start=START_IDX):
    # 현재 문제의 테스트케이스들 추출 (마스크 사용)
    mask = df_testcases['problem_id'] == prob_id
    group = df_testcases[mask]
    
    if group.empty:
        # [추가된 출력] 조용히 건너뛰지 않고 원인과 함께 경고 메시지 출력
        print(f"\n  ⚠️ [{idx}번째 문제] Problem {prob_id} (테스트케이스 데이터를 찾을 수 없어 건너뜁니다!)")
        continue
        
    total_tcs = len(group)
    print(f"\n  📝 [{idx}번째 문제] Problem {prob_id} (총 테스트케이스: {total_tcs}개) 처리 시작...")
    
    # [조건 1] 이미 type이 모두 비어있지 않고, ERROR가 아니라면 완벽히 처리된 것으로 간주하고 패스
    if group['type'].notnull().all() and not (group['type'] == 'ERROR').any():
        print(f"    ⏭ 이미 분류가 완료된 문제입니다. (건너뜀)")
        continue

    # 50개 단위로 쪼개서 배치 처리
    for i in range(0, total_tcs, BATCH_SIZE):
        batch = group.iloc[i:i+BATCH_SIZE]
        
        tc_payload = []
        for _, row in batch.iterrows():
            tc_payload.append({
                "testcase_order": str(row['testcase_order']),
                "input": str(row['input']),
                "output": str(row['output'])
            })
        
        print(f"    - Testcases {batch['testcase_order'].iloc[0]} ~ {batch['testcase_order'].iloc[-1]} 분류 요청 중...")
        
        start_time = time.time()
        
        # LLM API 호출
        classification_result = classify_testcases_with_retry(question_text, tc_payload)
        
        elapsed_time = time.time() - start_time
        print(f"    ✔ 분류 완료! (소요 시간: {elapsed_time:.2f}초)")
        
        # 받아온 JSON 결과를 메인 데이터프레임에 즉시 매핑
        for tc_order, tc_type in classification_result.items():
            tc_mask = mask & (df_testcases['testcase_order'].astype(str) == str(tc_order))
            df_testcases.loc[tc_mask, 'type'] = tc_type

    # [조건 2] 하나의 문제(Problem)가 완전히 처리될 때마다 전체 데이터프레임을 덮어쓰기 (중간 저장)
    df_testcases.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
    print(f"  💾 Problem {prob_id}의 분류 결과 중간 저장 완료!")

print(f"\n🎉 설정한 범위({START_IDX}~{END_IDX}번째)의 테스트케이스 분류가 완료되었습니다!")

🚀 [Step 1] 문제 및 테스트케이스 데이터 로드 중...


/tmp/ipykernel_51649/1935017902.py:93: DtypeWarning: Columns (0: id, 1: count_cases, 2: count_solutions, 3: Expected Auxiliary Space, 4: starter_code, 5: picture_num, 6: Unnamed: 21, 7: Unnamed: 22, 8: Unnamed: 23, 9: Unnamed: 25, 10: Unnamed: 26, 11: Unnamed: 30, 12: Unnamed: 32, 13: Unnamed: 36, 14: Unnamed: 50, 15: Unnamed: 54, 16: Unnamed: 56, 17: Unnamed: 60, 18: Unnamed: 80, 19: Unnamed: 84, 20: Unnamed: 92, 21: Unnamed: 102, 22: Unnamed: 110, 23: Unnamed: 114, 24: Unnamed: 120, 25: Unnamed: 126, 26: Unnamed: 140, 27: Unnamed: 144, 28: Unnamed: 150, 29: Unnamed: 156, 30: Unnamed: 164, 31: Unnamed: 170, 32: Unnamed: 173) have mixed types. Specify dtype option on import or set low_memory=False.
  df_problems = pd.read_csv(PROBLEMS_FILE)


📄 원본 테스트케이스 파일(validated_problem_testcases.csv)을 새로 불러옵니다.
✅ 총 4634문제 중 1번째 ~ 4529번째 문제(총 4529개) 작업을 시작합니다.

🚀 [Step 2] 테스트케이스 분류 작업 시작...

  📝 [1번째 문제] Problem 2 (총 테스트케이스: 94개) 처리 시작...
    - Testcases 1 ~ 50 분류 요청 중...
    ✔ 분류 완료! (소요 시간: 95.55초)
    - Testcases 51 ~ 94 분류 요청 중...
      ⚠️ API 호출 에러(500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}). 1.5초 후 재시도합니다...
      ⚠️ API 호출 에러(500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}). 2.7초 후 재시도합니다...
    ✔ 분류 완료! (소요 시간: 109.47초)
  💾 Problem 2의 분류 결과 중간 저장 완료!

  📝 [2번째 문제] Problem 11 (총 테스트케이스: 78개) 처리 시작...
    - Testcases 1 ~ 50 분류 요청 중...
    ✔ 분류 완료! (소요 시간: 244.82초)
    - Testcases 51 ~ 78 분류 요청 중...
      ⚠️ API 호출 에러(500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}). 1.6초 후 재시도합니다...


KeyboardInterrupt: 